# Dropout

_Deep Learning_

**Randomly switch off neurons during training so the network can't lean on any one of them.**

Overfitting means the network memorizes the training data instead of learning general patterns.

     Dropout fights this. During training, it randomly turns off some neurons each step.

---

This notebook is a practice scaffold for the **Dropout** lesson. We build it up one small step at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

### Step 1 — Build a network with a dropout layer

We stack a linear layer, a ReLU, then an `nn.Dropout(p=0.5)` layer, then a final linear layer. The dropout layer is the key piece: on each forward pass during training it randomly zeros out half of the activations flowing through it, so the network can never rely on any single neuron.

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Dropout(p=0.5),        # drop 50% of neurons each forward pass
    nn.Linear(64, 1),
)

### Step 2 — Run a forward pass in training mode

We feed in a small batch of random inputs. Calling `net.train()` puts the network in **training mode**, which turns dropout ON — so each neuron is randomly kept or zeroed. Because the mask is random, re-running this cell gives slightly different outputs each time.

In [ ]:
x = torch.randn(4, 20)

net.train()                   # training mode: dropout ON (random zeros)
train_out = net(x).squeeze().round(decimals=3).tolist()
print("train out:", train_out)

### Step 3 — Run the same input in eval mode

Calling `net.eval()` switches to **evaluation mode**, which turns dropout OFF: every neuron is used and PyTorch rescales appropriately so the expected output matches training. The result is deterministic — this is the mode you use for validation and inference.

In [ ]:
net.eval()                    # eval mode: dropout OFF (use full network)
with torch.no_grad():
    eval_out = net(x).squeeze().round(decimals=3).tolist()
print("eval  out:", eval_out)

## Visualize the data & results

_Training on real tumor data, does strong regularization (dropout-style) close the train-validation gap?_

### Step 1 — Load and split the real tumor data

We load the breast-cancer dataset, standardise every feature (subtract the mean, divide by the std), and split it into a training set and a held-out test set. The split is **stratified** so both sets keep the same balance of malignant and benign cases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)
Xtr, Xte, ytr, yte = train_test_split(
    X, bc.target, test_size=0.3, random_state=0, stratify=bc.target
)

### Step 2 — Train two models at different regularization strengths

`MLPClassifier`'s `alpha` is an L2 weight penalty — a stand-in for dropout-style regularization. We define a helper that trains one network epoch by epoch (using `warm_start` + `partial_fit`) and records the train and validation log-loss after each epoch. We then run it twice: once with a tiny `alpha` (weak regularization, which overfits) and once with a large `alpha` (strong regularization).

In [ ]:
def curves(alpha):
    m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                      warm_start=True, random_state=0, alpha=alpha,
                      learning_rate_init=0.003)
    tr, va = [], []
    for e in range(40):
        if e == 0:
            m.partial_fit(Xtr, ytr, classes=[0, 1])
        else:
            m.partial_fit(Xtr, ytr)
        tr.append(log_loss(ytr, m.predict_proba(Xtr), labels=[0, 1]))
        va.append(log_loss(yte, m.predict_proba(Xte), labels=[0, 1]))
    return tr, va

tr_w, va_w = curves(1e-5)               # weak reg, overfits
_, va_s = curves(3.0)                   # strong reg (dropout-like)

### Step 3 — Plot the train vs validation curves

We draw the weak-regularization training and validation loss together with the strong-regularization validation loss. Watch the gap: with weak regularization the validation loss pulls away from the train loss (overfitting), while strong regularization keeps the validation loss lower and flatter.

In [ ]:
plt.plot(tr_w, color="#4ea1ff", label="train (weak reg)")
plt.plot(va_w, color="#ff7b72", label="val (weak reg, overfits)")
plt.plot(va_s, color="#7ee787", label="val (strong reg)")
plt.title("Real train vs validation log-loss on breast-cancer data")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.legend()
plt.show()